In [1]:
print("Checking for the NVIDIA GPU access.")
!nvidia-smi

Checking for the NVIDIA GPU access.
Wed Mar  4 15:13:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2060        Off |   00000000:29:00.0  On |                  N/A |
| 33%   44C    P0             44W /  184W |     746MiB /  12288MiB |     29%      Default |
|                                         |                        |                  N/A |
+-----------

In [2]:
import os
import sys
import torch
import transformers
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

In [3]:
from dotenv import load_dotenv
load_dotenv()

os.environ['HF_TOKEN']=os.getenv('HF_TOKEN')

In [4]:
model_checkpoint="Helsinki-NLP/opus-mt-en-hi"

`Model`:**Helsinki-NLP/opus-mt-en-hi**

`Checkout the model details` [here](https://huggingface.co/Helsinki-NLP/opus-mt-en-hi)

## Getting the Dataset

Repo name: `cfilt/iitb-english-hindi`

[Source](https://huggingface.co/datasets/cfilt/iitb-english-hindi)

In [8]:
raw_dataset = load_dataset("cfilt/iitb-english-hindi")

## Defining the number of samples to use for training
num_samples = len(raw_dataset['train']) // 10  # Using 10% of the training data for faster experimentation
print("Preparing the dataset...")
print(f"Proceding with {num_samples} samples for training.")

## Creating a subset of the training dataset for faster experimentation
train_subset = raw_dataset["train"].shuffle(seed=42).select(range(num_samples))
val_dataset = raw_dataset["validation"]
test_dataset = raw_dataset["test"]

print(f"Original Train Size: {len(raw_dataset['train'])}")
print(f"Subset Train Size: {len(train_subset)}")

Preparing the dataset...
Proceding with 165908 samples for training.
Original Train Size: 1659083
Subset Train Size: 165908


In [9]:
raw_dataset['train'][1]

{'translation': {'en': 'Accerciser Accessibility Explorer',
  'hi': 'एक्सेर्साइसर पहुंचनीयता अन्वेषक'}}

## Processing the Data

In [10]:
#Initializing the Tokenizer to embedd the data
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [11]:
## Checking the Tokenizer and generating the random embedd vector
tokenizer("Hello, This a tokenizer and it is working fine.")

{'input_ids': [12110, 2, 239, 19, 11608, 5985, 896, 10, 52, 23, 2336, 1501, 3, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [12]:
tokenizer(["Sunn is always rises in the east","Artificial Intelligence is helpful in current era."])

{'input_ids': [[11647, 813, 23, 1205, 16500, 21, 4, 5631, 0], [16178, 4454, 1826, 7319, 59188, 23, 6429, 21, 899, 13632, 3, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

In [13]:
## Generating embeddings for hindi sentence
tokenizer("एक्सेर्साइसर पहुंचनीयता अन्वेषक")

{'input_ids': [44, 1042, 716, 1185, 1960, 428, 1185, 680, 260, 7173, 680, 428, 44, 2703, 24217, 549, 4228, 314, 130, 12604, 260, 44, 4499, 314, 1185, 1, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [14]:
## Defining the preprocessing parameters and function

max_input_length=128
max_target_length=128

source_lang="en" # en:english
target_lang="hi" # hi:hindi

In [15]:
# data preprocessing function
def preprocess_function(examples):
  inputs = [ex[source_lang] for ex in examples["translation"]]
  targets = [ex[target_lang] for ex in examples["translation"]]
  model_inputs=tokenizer(inputs, max_length=max_input_length, truncation=True)

  # Tokenize targets directly, as MarianTokenizer does not have as_target_tokenizer
  with tokenizer.as_target_tokenizer():
    labels = tokenizer(targets, max_length=max_target_length, truncation=True)

  model_inputs["labels"] = labels["input_ids"]
  return model_inputs

## Calling the function to test few rows of dataset
preprocess_function(raw_dataset['train'][:2])

/home/prashant/.conda/envs/genai/lib/python3.13/site-packages/transformers/tokenization_utils_base.py:4174: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


{'input_ids': [[3872, 85, 2501, 132, 15441, 36398, 0], [32643, 28541, 36253, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1]], 'labels': [[63, 2025, 18, 16155, 346, 20311, 24, 2279, 679, 0], [26618, 16155, 346, 33383, 0]]}

In [19]:
train_tokenized_dataset=train_subset.map(preprocess_function, batched=True)

eval_tokenized_dataset=val_dataset.map(preprocess_function, batched=True)

print(train_tokenized_dataset[0])
print(eval_tokenized_dataset[0])

Map:   0%|          | 0/165908 [00:00<?, ? examples/s]

/home/prashant/.conda/envs/genai/lib/python3.13/site-packages/transformers/tokenization_utils_base.py:4174: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/520 [00:00<?, ? examples/s]

{'translation': {'en': 'on the intuition.', 'hi': 'अंतर्ज्ञान पर। '}, 'input_ids': [68, 4, 61720, 3, 0], 'attention_mask': [1, 1, 1, 1, 1], 'labels': [25130, 21126, 33, 40, 0]}
{'translation': {'en': 'Students of the Dattatreya city Municipal corporation secondary school demonstrated their imagination power by creating the fictitious fort "Duttgarh".', 'hi': "महानगर पालिका अंतर्गत दत्तात्रय नगर माध्यमिक स्कूल के विद्यार्थियों ने काल्पनिक किला 'दत्तगढ़' बनाकर अपनी कल्पनाशक्ति का परिचय दिया।"}, 'input_ids': [8089, 8, 4, 16826, 21235, 142, 661, 15908, 949, 12111, 19561, 9646, 916, 34734, 20470, 2108, 8885, 77, 17274, 763, 97, 6480, 4, 53372, 34, 142, 83, 687, 5082, 142, 11695, 2326, 1842, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [25359, 13400, 11409, 15772, 2448, 31545, 8469, 1321, 1409, 49935, 2055, 6, 5909, 37, 22976, 21307, 256, 3130, 14917, 48102, 70, 2385, 143, 4712, 13454, 24, 11160, 139, 

In [20]:
#Model training parameters
batch_size=16
learning_rate=2e-5
weight_decay=0.01
num_train_epochs=1


## Initialize Trainer

### Subtask:
Initialize the `Seq2SeqTrainer` with the loaded model, `training_args`, `train_dataset`, `validation_dataset`, and `data_collator`.


In [ ]:
model_name = model_checkpoint.split("/")[-1]

# 1. Load the model
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

# 2. Data Collator (handles dynamic padding)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 3. Define Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=f"{model_name}-finetuned-{source_lang}-to-{target_lang}",
    eval_strategy="epoch",
    # optimizer="adamw_torch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,  # Set to False if not using a GPU
    push_to_hub=False,
)

# 4. Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized_dataset, # Your 1 Lakh subset
    eval_dataset=eval_tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# 5. Start Training
trainer.train()

/tmp/ipykernel_6748/2931543314.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss
1,2.640300,3.517111
2,2.386400,3.375788
3,2.245200,3.338326


/home/prashant/.conda/envs/genai/lib/python3.13/site-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[61949]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=31110, training_loss=2.513403668552302, metrics={'train_runtime': 2652.7382, 'train_samples_per_second': 187.627, 'train_steps_per_second': 11.728, 'total_flos': 8141410559066112.0, 'train_loss': 2.513403668552302, 'epoch': 3.0})

## Model Testing

### 1. Manual Inference (The Quick Test)

In [27]:
from transformers import pipeline, AutoTokenizer, TFAutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-hi")

# Initialize the translation pipeline
translator = pipeline("translation_en_to_hi", model=model, tokenizer=tokenizer)

# Test sentences
sentences = [
    "I am learning how to train machine learning models.",
    "The capital of India is New Delhi.",
    "Can you help me with this task?"
]

for text in sentences:
    output = translator(text, max_length=40)
    print(f"English: {text}")
    print(f"Hindi:   {output[0]['translation_text']}\n")

Device set to use cuda:0


English: I am learning how to train machine learning models.
Hindi:   मैं सीख रहा हूँ कि मशीन सीखने मॉडल सीखने के लिए कैसे सीख रहा हूँ।

English: The capital of India is New Delhi.
Hindi:   भारत की राजधानी नई दिल्ली है.

English: Can you help me with this task?
Hindi:   आप इस कार्य के साथ मेरी मदद कर सकते हैं?



### 2. Systematic Evaluation (The Metric Test)

In [31]:
import tqdm

def generate_translations(batch_size=16):
    inputs = [ex["en"] for ex in test_dataset["translation"]]
    ground_truth = [ex["hi"] for ex in test_dataset["translation"]]
    predictions = []

    for i in tqdm.tqdm(range(0, len(inputs), batch_size)):
        batch = inputs[i : i + batch_size]
        # Tokenize and generate
        tokenized_batch = tokenizer(batch, padding=True, truncation=True,return_tensors="pt").to(model.device)
        out = model.generate(**tokenized_batch, max_length=128)
        
        # Decode back to text
        decoded_out = tokenizer.batch_decode(out, skip_special_tokens=True)
        predictions.extend(decoded_out)
        
    return predictions, ground_truth

preds, refs = generate_translations()

100%|██████████| 157/157 [01:29<00:00,  1.75it/s]


### 3. Compare with SacreBLEU

We have preds (what model said) and refs (what a human said), calculating the score:

In [34]:
import evaluate

sacrebleu = evaluate.load("sacrebleu")

# SacreBLEU expects references as a list of lists
formatted_refs = [[r] for r in refs]

results = sacrebleu.compute(predictions=preds, references=formatted_refs)
print(f"Final BLEU Score: {results['score']:.2f}")

Final BLEU Score: 9.89


In [39]:
import numpy as np

input_text  = "I love to learn new things and explore the world of artificial intelligence."

tokenized = tokenizer([input_text], return_tensors="pt").to(model.device)
out = model.generate(**tokenized, max_length=128)
print(out)

tensor([[61949,   104,  1794,   613,  6737,     9, 26437,  1428,    15,   463,
            15,   808,   187,  1346,   161,   254,     3,     0]],
       device='cuda:0')


In [40]:
with tokenizer.as_target_tokenizer():
    print(tokenizer.decode(out[0], skip_special_tokens=True))

मैं नयी बातें सीखना और कृत्रिम बुद्धि की दुनिया की खोज करना पसंद करता हूँ.


/home/prashant/.conda/envs/genai/lib/python3.13/site-packages/transformers/tokenization_utils_base.py:4174: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
